<a href="https://colab.research.google.com/github/HereLiesAz/CueDetat/blob/main/Copy_of_pocket_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cue d'Etat — Pocket Detector Training

A deconstructed, redundancy-free pipeline. It mounts, unzips, aggressively maps classes to achieve absolute parity, trains YOLOv8n, and exports to TFLite FP16 without the existential dread of missing labels.

In [16]:
!pip install -q ultralytics roboflow kagglehub

import os
import shutil
import yaml
from google.colab import drive

# Robust mounting: only mount if not already present
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted.")

PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

for d in [PROJECT_DIR, DATASETS_BASE, COMBINED_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Environment Ready. Project Directory: {PROJECT_DIR}")

Drive already mounted.
Environment Ready. Project Directory: /content/drive/MyDrive/billiards_training


In [ ]:
import os

local_zip = '/content/billiards_training.zip'

if os.path.exists(local_zip):
    print(f"✅ Verified: {local_zip} is ready in local content root.")
    print(f"Size: {os.path.getsize(local_zip) / (1024**2):.2f} MB")
else:
    print(f"❌ Critical: {local_zip} not found. Please upload it or ensure the path is correct.")

In [17]:
import kagglehub
path = kagglehub.dataset_download("hereliesaz/cue-detat")

Using Colab cache for faster access to the 'cue-detat' dataset.


In [18]:
import kagglehub
path = kagglehub.dataset_download("diveshcrazy/pool-table-balls-classification")

Using Colab cache for faster access to the 'pool-table-balls-classification' dataset.


In [19]:
import os
import shutil
import yaml
from tqdm.auto import tqdm

# Define paths from previous cells and kernel state
KAGGLE_BALLS_PATH = '/root/.cache/kagglehub/datasets/diveshcrazy/pool-table-balls-classification/versions/1'
KAGGLE_CUE_PATH = '/root/.cache/kagglehub/datasets/hereliesaz/cue-detat/versions/1'
COMBINED_DIR = '/content/local_combined_dataset'
master_classes = ['pool-table', 'pool-table-hole', 'pool-table-side']

# Ensure base directories exist
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(COMBINED_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(COMBINED_DIR, split, 'labels'), exist_ok=True)

def merge_kaggle_source(source_path, name_tag):
    print(f"Processing Kaggle Source: {name_tag}")
    for root, dirs, files in os.walk(source_path):
        if 'data.yaml' in files:
            with open(os.path.join(root, 'data.yaml'), 'r') as f:
                cfg = yaml.safe_load(f)
                names = cfg.get('names', [])
                if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]
                # Map source classes to master indices
                cmap = {i: master_classes.index(n) for i, n in enumerate(names) if n in master_classes}

            # Find images
            for r, _, f_list in os.walk(root):
                if 'images' in r:
                    split = 'valid' if 'val' in r or 'valid' in r else 'test' if 'test' in r else 'train'
                    images = [f for f in f_list if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

                    for f in tqdm(images, desc=f"  Merging {name_tag} ({split})"):
                        # Use name_tag to prevent filename collisions
                        unique_name = f"{name_tag}_{f}"
                        shutil.copy2(os.path.join(r, f), os.path.join(COMBINED_DIR, split, 'images', unique_name))

                        # Handle labels
                        lbl = os.path.splitext(f)[0] + '.txt'
                        lbl_path = os.path.join(r.replace('images', 'labels'), lbl)
                        if os.path.exists(lbl_path):
                            with open(lbl_path, 'r') as f_in, open(os.path.join(COMBINED_DIR, split, 'labels', os.path.splitext(unique_name)[0] + '.txt'), 'w') as f_out:
                                for line in f_in:
                                    p = line.split()
                                    if p and int(p[0]) in cmap:
                                        p[0] = str(cmap[int(p[0])])
                                        f_out.write(' '.join(p) + '\n')

# Process both downloads
merge_kaggle_source(KAGGLE_BALLS_PATH, 'balls_ds')
merge_kaggle_source(KAGGLE_CUE_PATH, 'cue_ds')

# Update data.yaml
with open(os.path.join(COMBINED_DIR, 'data.yaml'), 'w') as f:
    yaml.dump({
        'train': os.path.join(COMBINED_DIR, 'train', 'images'),
        'val': os.path.join(COMBINED_DIR, 'valid', 'images'),
        'nc': len(master_classes),
        'names': master_classes
    }, f)

print("\nKaggle sources merged and remapped successfully.")

Processing Kaggle Source: balls_ds
Processing Kaggle Source: cue_ds


  Merging cue_ds (train):   0%|          | 0/2688 [00:00<?, ?it/s]

  Merging cue_ds (valid):   0%|          | 0/284 [00:00<?, ?it/s]

  Merging cue_ds (test):   0%|          | 0/262 [00:00<?, ?it/s]

  Merging cue_ds (train):   0%|          | 0/37222 [00:00<?, ?it/s]

  Merging cue_ds (valid):   0%|          | 0/5664 [00:00<?, ?it/s]

  Merging cue_ds (test):   0%|          | 0/3379 [00:00<?, ?it/s]

  Merging cue_ds (train):   0%|          | 0/960 [00:00<?, ?it/s]

  Merging cue_ds (valid):   0%|          | 0/240 [00:00<?, ?it/s]


Kaggle sources merged and remapped successfully.


In [ ]:
import os
from google.colab import drive

# Ensure drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define the global project structure constants
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
COMBINED_DIR = os.path.join(DATASETS_BASE, 'combined_dataset')

# Create missing directories
for d in [PROJECT_DIR, DATASETS_BASE, COMBINED_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Paths initialized:\nProject: {PROJECT_DIR}\nDatasets: {DATASETS_BASE}\nCombined: {COMBINED_DIR}")

In [1]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

export_path = os.path.join(PROJECT_DIR, 'exports', 'pocket_detector_fp16.tflite')
if os.path.exists(export_path):
    interp = tf.lite.Interpreter(model_path=export_path)
    interp.allocate_tensors()
    inp_d, out_d = interp.get_input_details(), interp.get_output_details()
    SZ, dtype = int(inp_d[0]['shape'][1]), inp_d[0]['dtype']

    def infer(path):
        rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
        interp.set_tensor(inp_d[0]['index'], inp)
        interp.invoke()
        return rgb, interp.get_tensor(out_d[0]['index'])[0], rgb.shape[1], rgb.shape[0]

    imgs = glob.glob(os.path.join(COMBINED_DIR, 'valid', 'images', '*.jpg'))[:4]
    if imgs:
        fig, axes = plt.subplots(1, len(imgs), figsize=(15, 5))
        if len(imgs) == 1: axes = [axes]

        for ax, p in zip(axes, imgs):
            img, dets, w, h = infer(p)
            draw = img.copy()
            n = 0
            # YOLOv8 TFLite output shape varies; typical is [1, 6, 8400]
            # This is a simplified visualization loop
            ax.imshow(draw); ax.set_title(f"Inference complete"); ax.axis('off')
        plt.show()
    else:
        print("No validation images found for smoke test.")
else:
    print(f"TFLite model not found at {export_path}")

NameError: name 'os' is not defined

In [2]:
import os

# Ensure paths are defined
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')

if not os.path.exists(DATASETS_BASE):
    os.makedirs(DATASETS_BASE, exist_ok=True)

# Scanning the datasets directory for additional ZIP files
found_zips = [f for f in os.listdir(DATASETS_BASE) if f.endswith('.zip')]
print(f'Found ZIPs in {DATASETS_BASE}: {found_zips}')

# Also scanning the Drive root just in case they are there
drive_zips = [f for f in os.listdir('/content/drive/MyDrive') if f.endswith('.zip') and any(term in f.lower() for term in ['pool', 'billiards', 'training'])]
print(f'Found relevant ZIPs in Drive root: {drive_zips}')

Found ZIPs in /content/drive/MyDrive/billiards_training/datasets: []
Found relevant ZIPs in Drive root: []


In [3]:
# Dynamic Merge, Label Fix & Cleanup
import os
import yaml
import shutil

# Automatically find all subdirectories in DATASETS_BASE excluding the combined_dataset itself
sources = [os.path.join(DATASETS_BASE, d) for d in os.listdir(DATASETS_BASE)
           if os.path.isdir(os.path.join(DATASETS_BASE, d)) and d != 'combined_dataset']

print(f"Merging from sources: {sources}")

master_classes = ['pool-table', 'pool-table-hole', 'pool-table-side']

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(COMBINED_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(COMBINED_DIR, split, 'labels'), exist_ok=True)

label_lookup, img_lookup = {}, {}
for base in sources:
    if not os.path.exists(base): continue
    for root, _, files in os.walk(base):
        if 'data.yaml' in files:
            with open(os.path.join(root, 'data.yaml'), 'r') as f:
                data_cfg = yaml.safe_load(f)
                names = data_cfg.get('names', [])
                if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]
                cmap = {i: master_classes.index(n) for i, n in enumerate(names) if n in master_classes}

            for r, _, f_list in os.walk(root):
                split_guess = 'valid' if 'val' in r or 'valid' in r else 'test' if 'test' in r else 'train'
                if 'labels' in r:
                    for f_name in f_list:
                        if f_name.endswith('.txt'): label_lookup[f_name] = (os.path.join(r, f_name), cmap, split_guess)
                if 'images' in r:
                    for f_name in f_list:
                        if f_name.lower().endswith(('.jpg', '.png', '.jpeg')): img_lookup[f_name] = (os.path.join(r, f_name), split_guess)

for img_name, (img_path, split) in img_lookup.items():
    shutil.copy2(img_path, os.path.join(COMBINED_DIR, split, 'images', img_name))
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    if lbl_name in label_lookup:
        src_path, cmap, _ = label_lookup[lbl_name]
        with open(src_path, 'r') as f_in, open(os.path.join(COMBINED_DIR, split, 'labels', lbl_name), 'w') as f_out:
            for line in f_in:
                parts = line.split()
                if parts and int(parts[0]) in cmap:
                    parts[0] = str(cmap[int(parts[0])])
                    f_out.write(' '.join(parts) + '\n')

with open(os.path.join(COMBINED_DIR, 'data.yaml'), 'w') as f:
    yaml.dump({'train': os.path.join(COMBINED_DIR, 'train', 'images'), 'val': os.path.join(COMBINED_DIR, 'valid', 'images'), 'test': os.path.join(COMBINED_DIR, 'test', 'images'), 'nc': len(master_classes), 'names': master_classes}, f)

print("Merge and cleanup complete.")

Merging from sources: ['/content/drive/MyDrive/billiards_training/datasets/billiards_training']


NameError: name 'COMBINED_DIR' is not defined

In [4]:
import shutil, os, yaml
LOCAL_COMBINED = '/content/local_combined_dataset'
os.makedirs(LOCAL_COMBINED, exist_ok=True)
print(f'Syncing dataset to {LOCAL_COMBINED}...')
# Simple recursive copy for local performance
if os.path.exists(COMBINED_DIR):
    !cp -r {COMBINED_DIR}/* {LOCAL_COMBINED}/
# Correct paths in local data.yaml
local_yaml = os.path.join(LOCAL_COMBINED, 'data.yaml')
if os.path.exists(local_yaml):
    with open(local_yaml, 'r') as f: cfg = yaml.safe_load(f)
    cfg['train'] = os.path.join(LOCAL_COMBINED, 'train', 'images')
    cfg['val'] = os.path.join(LOCAL_COMBINED, 'valid', 'images')
    cfg['test'] = os.path.join(LOCAL_COMBINED, 'test', 'images')
    with open(local_yaml, 'w') as f: yaml.dump(cfg, f)
print('Local sync complete.')

Syncing dataset to /content/local_combined_dataset...


NameError: name 'COMBINED_DIR' is not defined

In [ ]:
# Export to TFLite FP16 for deployment
print("Exporting model...")
export_path = model.export(format='tflite', imgsz=640, half=True, nms=True)

# Move to exports folder
export_dir = os.path.join(PROJECT_DIR, 'exports')
os.makedirs(export_dir, exist_ok=True)
target_file = os.path.join(export_dir, 'pocket_detector_fp16.tflite')
shutil.copy2(export_path, target_file)
print(f"Model exported to: {target_file}")

In [5]:
import torch
import os

ckpt_path = '/content/drive/MyDrive/billiards_training/pocket_detector_final_v13/weights/last.pt'

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    print("--- Detailed Checkpoint Metadata ---")
    if 'model' in ckpt:
        m = ckpt['model']
        # Accessing class info directly from the model or its dictionary
        nc = getattr(m, 'nc', 'Unknown')
        names = getattr(m, 'names', 'Unknown')
        print(f"Model Number of Classes (nc): {nc}")
        print(f"Model Class Names: {names}")

        # Check if the parameter groups are available for inspection
        if 'optimizer' in ckpt:
            print(f"Optimizer Parameter Groups: {len(ckpt['optimizer'].get('param_groups', []))}")

    if 'train_args' in ckpt:
        args = ckpt['train_args']
        data_path = args.get('data') if isinstance(args, dict) else getattr(args, 'data', 'N/A')
        print(f"Checkpoint Dataset YAML: {data_path}")

    print(f"Epochs completed: {ckpt.get('epoch', 'N/A')}")
else:
    print("Checkpoint not found.")

Checkpoint not found.


In [ ]:
from datetime import datetime
report_path = os.path.join(PROJECT_DIR, 'training_report.md')
content = f"# Training Report\nGenerated: {datetime.now()}\n- Classes: {master_classes}\n- Export: pocket_detector_fp16.tflite"
with open(report_path, 'w') as f: f.write(content)

# Cleanup redundant sources in Drive to save space
for d in os.listdir(DATASETS_BASE):
    path = os.path.join(DATASETS_BASE, d)
    if os.path.isdir(path) and d != 'combined_dataset':
        shutil.rmtree(path)
print("Cleanup complete and report generated.")

In [6]:
import os
import shutil
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
REPO_URL = 'https://github.com/HereLiesAz/CueDetat.git'
REPO_DIR = '/content/CueDetat'
DRIVE_BACKUP = '/content/drive/MyDrive/CueDetat_Backup'

# 3. Clone Repository if it doesn't exist
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

# 4. Create Drive backup directory
os.makedirs(DRIVE_BACKUP, exist_ok=True)

print("Environment initialized.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'CueDetat'...
remote: Enumerating objects: 16525, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 16525 (delta 7), reused 19 (delta 5), pack-reused 16479 (from 3)
Receiving objects: 100% (16525/16525), 430.16 MiB | 31.55 MiB/s, done.
Resolving deltas: 100% (8957/8957), done.
Updating files: 100% (408/408), done.
Environment initialized.


In [7]:
import os
import shutil

# 1. Unzip the local file
DATASET_ZIP = '/content/billiards_training.zip'
LOCAL_EXTRACT = '/content/datasets_extracted'

if os.path.exists(DATASET_ZIP):
    print(f"Unzipping {DATASET_ZIP}...")
    os.makedirs(LOCAL_EXTRACT, exist_ok=True)
    !unzip -qo {DATASET_ZIP} -d {LOCAL_EXTRACT}
    print("Unzip complete.")
else:
    print("❌ Local 'billiards_training.zip' not found in /content/")

Unzipping /content/billiards_training.zip...
[/content/billiards_training.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/billiards_training.zip or
        /content/billiards_training.zip.zip, and cannot find /content/billiards_training.zip.ZIP, period.
Unzip complete.


In [8]:
import shutil
import os

LOCAL_DATASET = '/content/local_combined_dataset'
os.makedirs(LOCAL_DATASET, exist_ok=True)

print(f'Syncing dataset from Drive to {LOCAL_DATASET}...')

def sync_with_progress(src_root, dst_root):
    print("Calculating total files...")
    all_files = []
    for root, dirs, files in os.walk(src_root):
        for f in files:
            all_files.append(os.path.join(root, f))

    total = len(all_files)
    print(f"Total files to sync: {total}")

    copied = 0
    skipped = 0

    for src_path in all_files:
        rel_path = os.path.relpath(src_path, src_root)
        dst_path = os.path.join(dst_root, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)

        if os.path.exists(dst_path) and os.path.getsize(src_path) == os.path.getsize(dst_path):
            skipped += 1
        else:
            shutil.copy2(src_path, dst_path)

        copied += 1
        if copied % 50 == 0 or copied == total:
            print(f'\rProgress: {copied}/{total} ({(copied/total)*100:.1f}%) | Skipped: {skipped}', end='')

    print('\nSync complete. Starting comparative check...')

    # Verification Logic
    dst_files = []
    for root, dirs, files in os.walk(dst_root):
        for f in files: dst_files.append(os.path.join(root, f))

    src_size = sum(os.path.getsize(f) for f in all_files)
    dst_size = sum(os.path.getsize(f) for f in dst_files)

    print(f"Verification Results:")
    print(f"- Source: {len(all_files)} files, {src_size/1e6:.2f} MB")
    print(f"- Local:  {len(dst_files)} files, {dst_size/1e6:.2f} MB")

    if len(all_files) == len(dst_files) and src_size == dst_size:
        print("✅ Integrity check passed: Datasets are identical.")
    else:
        print("❌ Integrity check failed: Mismatch detected!")

sync_with_progress(COMBINED_DIR, LOCAL_DATASET)

local_yaml = os.path.join(LOCAL_DATASET, 'data.yaml')
if os.path.exists(local_yaml):
    import yaml
    with open(local_yaml, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['train'] = os.path.join(LOCAL_DATASET, 'train', 'images')
    cfg['val'] = os.path.join(LOCAL_DATASET, 'valid', 'images')
    cfg['test'] = os.path.join(LOCAL_DATASET, 'test', 'images')
    with open(local_yaml, 'w') as f:
        yaml.dump(cfg, f)

print('Local dataset is ready.')

Syncing dataset from Drive to /content/local_combined_dataset...


NameError: name 'COMBINED_DIR' is not defined

In [ ]:
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.4f}")

export_dir = os.path.join(PROJECT_DIR, 'exports')
os.makedirs(export_dir, exist_ok=True)
exported = model.export(format='tflite', imgsz=640, half=True, nms=True)

target = os.path.join(export_dir, 'pocket_detector_fp16.tflite')
if isinstance(exported, str):
    if exported.endswith('.tflite'):
        shutil.copy2(exported, target)
    elif os.path.isdir(exported):
        files = [f for f in os.listdir(exported) if f.endswith('.tflite')]
        if files: shutil.copy2(os.path.join(exported, files[0]), target)


In [ ]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

export_path = os.path.join(PROJECT_DIR, 'exports', 'pocket_detector_fp16.tflite')
interp = tf.lite.Interpreter(model_path=export_path)
interp.allocate_tensors()
inp_d, out_d = interp.get_input_details(), interp.get_output_details()
SZ, dtype = int(inp_d[0]['shape'][1]), inp_d[0]['dtype']

def infer(path):
    rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
    interp.set_tensor(inp_d[0]['index'], inp)
    interp.invoke()
    return rgb, interp.get_tensor(out_d[0]['index'])[0], rgb.shape[1], rgb.shape[0]

imgs = glob.glob(os.path.join(COMBINED_DIR, 'valid', 'images', '*.jpg'))[:4]
fig, axes = plt.subplots(1, max(len(imgs), 1), figsize=(15, 5))
if len(imgs) == 1: axes = [axes]

for ax, p in zip(axes, imgs):
    img, dets, w, h = infer(p)
    draw, n = img.copy(), 0
    for d in dets:
        conf, cls = float(d[4]) if len(d)>=5 else 0, int(d[5]) if len(d)>=6 else 0
        if conf >= 0.5 and cls == 1: # 1 is pocket
            x1, y1, x2, y2 = int(d[0]*w/SZ), int(d[1]*h/SZ), int(d[2]*w/SZ), int(d[3]*h/SZ)
            cv2.rectangle(draw, (x1,y1), (x2,y2), (0,255,0), 2)
            n += 1
    ax.imshow(draw); ax.set_title(f"{n} pockets"); ax.axis('off')
plt.show()


In [ ]:
weights = os.path.join(PROJECT_DIR, 'pocket_detector', 'weights')
backup = os.path.join(PROJECT_DIR, 'snapshots_backup')
if os.path.exists(weights):
    if os.path.exists(backup): shutil.rmtree(backup)
    shutil.copytree(weights, backup)

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    if os.path.exists('.git'):
        os.system('git add . && git commit -m "Auto-update weights" && git push')


In [ ]:
import os

PROJECT_DIR = '/content/drive/MyDrive/billiards_training'

if os.path.exists(PROJECT_DIR):
    print(f"Scanning {PROJECT_DIR} for trained models...\n")
    runs = [d for d in os.listdir(PROJECT_DIR) if os.path.isdir(os.path.join(PROJECT_DIR, d)) and 'pocket_detector' in d]

    if not runs:
        print("No training runs found.")
    else:
        for run in sorted(runs):
            weight_path = os.path.join(PROJECT_DIR, run, 'weights', 'best.pt')
            status = "✅ Weights Found" if os.path.exists(weight_path) else "❌ No weights yet"
            print(f"- Run: {run} | {status}")

    # Also check exports folder
    export_dir = os.path.join(PROJECT_DIR, 'exports')
    if os.path.exists(export_dir):
        print(f"\nScanning exports folder:")
        exports = [f for f in os.listdir(export_dir) if f.endswith(('.tflite', '.pt'))]
        for e in exports:
            print(f"- Exported Model: {e}")
else:
    print(f"Project directory {PROJECT_DIR} not found.")

In [ ]:
import os
import shutil

# Define paths
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
DATASETS_BASE = os.path.join(PROJECT_DIR, 'datasets')
EXTRACT_DIR = '/content/billiards_source_data'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# 1. Search for all relevant ZIP files in Drive
print("Searching for pool/billiards datasets in Drive...")
found_zips = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.endswith('.zip') and any(term in f.lower() for term in ['pool', 'billiards', 'training']):
            found_zips.append(os.path.join(root, f))

# 2. Extract every found dataset
if not found_zips:
    print("No relevant ZIP files found in Drive.")
else:
    for zip_path in found_zips:
        folder_name = os.path.splitext(os.path.basename(zip_path))[0]
        target_folder = os.path.join(EXTRACT_DIR, folder_name)
        print(f"Extracting {os.path.basename(zip_path)} to local storage...")
        os.makedirs(target_folder, exist_ok=True)
        !unzip -qo "{zip_path}" -d "{target_folder}"

print(f"Total datasets staged for merging: {len(os.listdir(EXTRACT_DIR))}")

In [9]:
import os

# Redefining paths to ensure scope safety
EXTRACT_DIR = '/content/billiards_source_data'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# Define the local ZIP files found in /content/
local_zips = [
    '/content/billiards_training.zip',
    '/content/pocket_detector2-20260327T125000Z-1-001.zip'
]

for zip_path in local_zips:
    if os.path.exists(zip_path):
        folder_name = os.path.splitext(os.path.basename(zip_path))[0]
        target_folder = os.path.join(EXTRACT_DIR, folder_name)
        print(f"Extracting local {os.path.basename(zip_path)} to {target_folder}...")
        os.makedirs(target_folder, exist_ok=True)
        !unzip -qo "{zip_path}" -d "{target_folder}"
    else:
        print(f"⚠️ Warning: {zip_path} not found in local root.")

print(f"Updated staging area. Total source folders: {len(os.listdir(EXTRACT_DIR))}")

Extracting local billiards_training.zip to /content/billiards_source_data/billiards_training...
[/content/billiards_training.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/billiards_training.zip or
        /content/billiards_training.zip.zip, and cannot find /content/billiards_training.zip.ZIP, period.
Extracting local pocket_detector2-20260327T125000Z-1-001.zip to /content/billiards_source_data/pocket_detector2-20260327T125000Z-1-001...
Updated staging area. Total source folders: 2


In [32]:
import os
import shutil
import yaml

# Verified paths from the global search
DRIVE_SRC = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset'
LOCAL_DIR = '/content/local_combined_dataset'

print(f'--- Syncing Data from Drive to Local Storage ---')

for split in ['train', 'valid', 'test']:
    src_img = os.path.join(DRIVE_SRC, split, 'images')
    src_lbl = os.path.join(DRIVE_SRC, split, 'labels')

    dst_img = os.path.join(LOCAL_DIR, split, 'images')
    dst_lbl = os.path.join(LOCAL_DIR, split, 'labels')

    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    if os.path.exists(src_img):
        print(f'Syncing {split} images...')
        # Use cp -r for speed in Colab
        !cp -r "{src_img}"/* "{dst_img}/"
        !cp -r "{src_lbl}"/* "{dst_lbl}/"
    else:
        print(f'⚠️ Warning: Split {split} not found in {DRIVE_SRC}')

# Create the data.yaml
master_classes = ['pool-table', 'pool-table-hole', 'pool-table-side']
with open(os.path.join(LOCAL_DIR, 'data.yaml'), 'w') as f:
    yaml.dump({
        'train': os.path.join(LOCAL_DIR, 'train', 'images'),
        'val': os.path.join(LOCAL_DIR, 'valid', 'images'),
        'nc': len(master_classes),
        'names': master_classes
    }, f)

print('\n✅ Sync complete. Local dataset is ready for training.')

--- Syncing Data from Drive to Local Storage ---
Syncing train images...
Syncing valid images...
Syncing test images...

✅ Sync complete. Local dataset is ready for training.


In [1]:
import os
import torch
from ultralytics import YOLO
from datetime import datetime

# 1. Configuration - Increased for better GPU performance
device = 0 if torch.cuda.is_available() else "cpu"
PROJECT_DIR = '/content/drive/MyDrive/billiards_training'
LOCAL_DATASET_YAML = '/content/local_combined_dataset/data.yaml'
TRAINING_NAME = 'pocket_detector_v13_fixed'

os.makedirs(PROJECT_DIR, exist_ok=True)

# 2. Dynamic Checkpoint Discovery
latest_checkpoint = None
latest_mtime = 0

print("🔍 Scanning Drive for the most recent checkpoint...")
for root, dirs, files in os.walk(PROJECT_DIR):
    if 'last.pt' in files:
        ckpt_path = os.path.join(root, 'last.pt')
        mtime = os.path.getmtime(ckpt_path)
        if mtime > latest_mtime:
            latest_mtime = mtime
            latest_checkpoint = ckpt_path

# 3. Model Initialization & Optimizer Fix
if latest_checkpoint:
    print(f"✅ Found checkpoint: {latest_checkpoint}")
    ckpt = torch.load(latest_checkpoint, map_location='cpu', weights_only=False)
    if 'optimizer' in ckpt:
        ckpt['optimizer'] = None
        fixed_ckpt_path = '/content/last_fixed.pt'
        torch.save(ckpt, fixed_ckpt_path)
        print("🛠️ Optimizer state stripped to prevent resume error.")
        model = YOLO(fixed_ckpt_path)
    else:
        model = YOLO(latest_checkpoint)
    resume_flag = False
else:
    print("🆕 No checkpoint found. Starting fresh with yolov8n.pt")
    model = YOLO('yolov8n.pt')
    resume_flag = False

# 4. Training Execution - High Performance Settings
results = model.train(
    data=LOCAL_DATASET_YAML,
    epochs=100,
    imgsz=640,
    batch=32,          # Increased for better GPU utilization
    workers=4,          # Increased workers for faster data loading
    project=PROJECT_DIR,
    name=TRAINING_NAME,
    device=device,
    resume=resume_flag,
    save=True,
    save_period=2,
    cache=True,        # Re-enabled cache for speed
    amp=True,
    exist_ok=True
)

print("\n✨ Block complete. Run again to continue.")

🔍 Scanning Drive for the most recent checkpoint...
✅ Found checkpoint: /content/drive/MyDrive/billiards_training/pocket_detector_v13_fixed/weights/last.pt
🛠️ Optimizer state stripped to prevent resume error.
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/local_combined_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask

KeyboardInterrupt: 

In [31]:
import os

print('--- Global Image Search ---')
found_data = False
for root, dirs, files in os.walk('/content'):
    # Skip system/git folders to be efficient
    if any(x in root for x in ['.config', '.git', 'sample_data']): continue

    # Look for .jpg files specifically (common for datasets)
    jpgs = [f for f in files if f.lower().endswith('.jpg')]
    if jpgs:
        print(f'\n📍 Found {len(jpgs)} JPGs in: {root}')
        print(f'   Sample: {jpgs[:2]}')
        found_data = True

        # Check for label presence
        txts = [f for f in files if f.endswith('.txt')]
        if txts:
            print(f'   ✅ {len(txts)} labels in same folder.')
        else:
            lbl_folder = os.path.join(os.path.dirname(root), 'labels')
            if os.path.exists(lbl_folder):
                print(f'   ✅ Labels found in sibling folder: {lbl_folder}')

if not found_data:
    print('❌ No JPG images found in /content. The ZIPs might contain a different format or extraction failed.')

--- Global Image Search ---

📍 Found 13 JPGs in: /content/drive/.shortcut-targets-by-id/1R7YdwelKK5YyfURbKQ4XUPUnMRlvjSFc/billiards_training/pocket_detector
   Sample: ['train_batch1.jpg', 'labels.jpg']

📍 Found 13 JPGs in: /content/drive/.shortcut-targets-by-id/1R7YdwelKK5YyfURbKQ4XUPUnMRlvjSFc/billiards_training/pocket_detector4
   Sample: ['labels.jpg', 'train_batch0.jpg']

📍 Found 7 JPGs in: /content/drive/.shortcut-targets-by-id/1R7YdwelKK5YyfURbKQ4XUPUnMRlvjSFc/billiards_training/pocket_detector6
   Sample: ['labels.jpg', 'train_batch0.jpg']

📍 Found 4 JPGs in: /content/drive/.shortcut-targets-by-id/1R7YdwelKK5YyfURbKQ4XUPUnMRlvjSFc/billiards_training/pocket_detector5
   Sample: ['labels.jpg', 'train_batch2.jpg']

📍 Found 484 JPGs in: /content/drive/.shortcut-targets-by-id/1R7YdwelKK5YyfURbKQ4XUPUnMRlvjSFc/billiards_training/datasets/combined_dataset/valid/images
   Sample: ['3-21-3-100-_png_jpg.rf.d5ee66ee1b916f1036746e8b5520049f.jpg', 'IMG_6350_jpg.rf.dd81f0d0b83b0dc9948bf9f3ff

In [28]:
from google.colab import drive
import os
import shutil

# Forcefully clear the mount point if it's blocking the process
if os.path.exists('/content/drive'):
    print("Cleaning up existing drive directory...")
    try:
        !fusermount -u /content/drive
    except:
        pass
    # If it's just a folder containing files, remove it
    if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
        shutil.rmtree('/content/drive')

# Attempt mount
drive.mount('/content/drive', force_remount=True)

Cleaning up existing drive directory...
Mounted at /content/drive


In [2]:
import os
from datetime import datetime

report_path = os.path.join(PROJECT_DIR, 'training_report.md')

# Gather Dataset Info
extracted_datasets = [d for d in os.listdir(DATASETS_BASE) if os.path.isdir(os.path.join(DATASETS_BASE, d))]

report_content = f"""# Cue d'Etat: Training Analysis Report
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 1. Dataset Inventory
- **Project Root:** `{PROJECT_DIR}`
- **Combined Dataset:** `{COMBINED_DIR}`
- **Source Datasets Detected:**
  - {f'\n  - '.join(extracted_datasets)}
- **Kaggle Source:** `{kaggle_path if 'kaggle_path' in globals() else 'N/A'}`

## 2. Model Configuration
- **Architecture:** YOLOv8n
- **Target Classes:** {master_classes}
- **Input Resolution:** 640x640
- **Training Schedule:** 100 Epochs / Batch 32

## 3. Artifact Locations
- **Final Weights:** `{os.path.join(PROJECT_DIR, TRAINING_NAME, 'weights', 'best.pt')}`
- **TFLite Export:** `{os.path.join(PROJECT_DIR, 'exports', 'pocket_detector_fp16.tflite')}`

## 4. Automated Review & Analysis
"""

def run_analysis():
    analysis = "### Model Performance Critique\n"
    if 'metrics' in globals():
        map50 = metrics.box.map50
        analysis += f"- **mAP50:** {map50:.4f}\n"
        if map50 < 0.7:
            analysis += "- **Warning:** Low mAP detected. Consider data augmentation or increasing class-specific samples for 'pocket-hole'.\n"
        else:
            analysis += "- **Status:** High confidence model. Ready for edge deployment.\n"
    else:
        analysis += "- *Note: Training metrics not found in current session. Run the training/export cells to populate analysis.*\n"

    analysis += "\n### Optimization Suggestions\n"
    analysis += "1. **Class Balancing:** Check if 'pool-table-side' is over-represented vs 'pool-table-hole'.\n"
    analysis += "2. **TFLite Quantization:** For mobile deployment, consider INT8 quantization if FP16 latency is > 50ms.\n"
    analysis += "3. **Synthetic Data:** Add motion-blurred frames to improve robustness against fast cue shots.\n"
    return analysis

report_content += run_analysis()

with open(report_path, 'w') as f:
    f.write(report_content)

print(f'Report generated successfully at: {report_path}')

NameError: name 'DATASETS_BASE' is not defined

In [ ]:
import os
import shutil

# Immediate cleanup of redundant datasets
sources_to_check = [os.path.join(DATASETS_BASE, d) for d in os.listdir(DATASETS_BASE) if os.path.isdir(os.path.join(DATASETS_BASE, d)) and d != 'combined_dataset']

print(f'Starting cleanup for {len(sources_to_check)} sources...')

for src in sources_to_check:
    if not os.path.exists(src): continue

    # Count files to verify there's data in the source
    file_count = sum([len(files) for r, d, files in os.walk(src)])

    if file_count > 0:
        print(f'Deleting verified source: {src} ({file_count} files)')
        try:
            shutil.rmtree(src)
        except Exception as e:
            print(f'Error deleting {src}: {e}')
    else:
        print(f'Skipping empty or missing directory: {src}')

print('Cleanup process finished.')